# **1. Decision Tree:** Overfitting vs Underfitting
Dataset: Iris Dataset (UCI/Kaggle)
Download:
https://www.kaggle.com/datasets/uciml/iris
# **Tasks**

1.Load the dataset.

2.Split it into 80% training and 20% testing data.

3.Train two Decision Tree models:

    Model A: max_depth=2

    Model B: max_depth=None

4.Compare:

    Training Accuracy

    Testing Accuracy

5.Which model is underfitting? Which one is overfitting?


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split




from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import accuracy_score,precision_score,recall_score

from sklearn.metrics import confusion_matrix,classification_report

from sklearn.model_selection import GridSearchCV

In [3]:
# 1 NO
from sklearn.datasets import load_iris

df = load_iris(as_frame=True).frame

df.sample(15)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
45,4.8,3.0,1.4,0.3,0
30,4.8,3.1,1.6,0.2,0
96,5.7,2.9,4.2,1.3,1
38,4.4,3.0,1.3,0.2,0
132,6.4,2.8,5.6,2.2,2
145,6.7,3.0,5.2,2.3,2
46,5.1,3.8,1.6,0.2,0
114,5.8,2.8,5.1,2.4,2
88,5.6,3.0,4.1,1.3,1
70,5.9,3.2,4.8,1.8,1


In [4]:
# 2 no
x=df.drop("target",axis=1)
y=df["target"]
x_train, x_test, y_train, y_test =train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
# 3 no
model_A=DecisionTreeClassifier(max_depth=2)
model_A.fit(x_train,y_train)



DecisionTreeClassifier(max_depth=2)

In [6]:
#training accuracy
y_pred_train=model_A.predict(x_train)
print(f"training accuracy :{accuracy_score(y_train,y_pred_train)}")


#testing accuracy

y_pred_test=model_A.predict(x_test)

print(f"testing accuracy :{accuracy_score(y_test,y_pred_test)}")


training accuracy :0.9666666666666667
testing accuracy :0.9333333333333333


In [7]:
# 3 no
model_A=DecisionTreeClassifier(max_depth=None)
model_A.fit(x_train,y_train)

DecisionTreeClassifier()

In [8]:
#training accuracy
y_pred_train=model_A.predict(x_train)
print(f"training accuracy :{accuracy_score(y_train,y_pred_train)}")


#testing accuracy

y_pred_test=model_A.predict(x_test)

print(f"testing accuracy :{accuracy_score(y_test,y_pred_test)}")

training accuracy :1.0
testing accuracy :0.9666666666666667


# 5 no

**Model B (`max_depth=None`)** → **Overfitting**

**কেন?**

* Training Accuracy = **100%**
* Testing Accuracy = **93.33%**

অর্থাৎ model training data **পুরোপুরি মুখস্থ** করে ফেলেছে। কিন্তু নতুন (testing) data-তে performance কমে গেছে।

এখানে gap হলো:

```
100% - 93.33% = 6.67%
```

এই gap-ই overfitting-এর লক্ষণ।

---

### Model A (`max_depth=2`)

* Training = **96.67%**
* Testing = **93.33%**

Gap:

```
96.67% - 93.33% = 3.34%
```

এখানে training accuracy-ও 100% নয়, কারণ tree ছোট হওয়ায় সব pattern শিখতে পারেনি। তাই এটিকে **slight underfitting** বা **কম complex model** বলা যায়।

---

### সহজ ভাষায়

ধরো ১০০টা প্রশ্ন দিয়ে একজন ছাত্রকে পড়ানো হলো।

* **Model A (`max_depth=2`)**

  * Training-এ 97টা ঠিক করেছে।
  * পরীক্ষায় 93টা ঠিক করেছে।
  * সবকিছু শেখেনি।

* **Model B (`max_depth=None`)**

  * Training-এ 100/100।
  * পরীক্ষায় 93/100।
  * Training-এর প্রশ্নগুলো মুখস্থ করেছে, তাই নতুন প্রশ্নে performance কমেছে।




* **Training খুব বেশি (≈100%) + Testing কম ⇒ Overfitting**
* **Training-ই কম ⇒ Underfitting**


# **2. Finding the Best Tree Depth**

Dataset: Wine Dataset
Download:

https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009

#Tasks

1.Load the dataset.

2.Train Decision Trees using:

    depth = 2

    depth = 4

    depth = 6

    depth = 8

    depth = None

3.Create a table containing:

    max_depth

    Train Accuracy



4.Which depth gives the best
generalization?


In [9]:
import pandas as pd

url = "https://raw.githubusercontent.com/plotly/datasets/master/winequality-red.csv"
df = pd.read_csv(url)

df.head(17)

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
5,7.4,0.660,0.00,1.8,0.075,13.0,40.0,0.9978,3.51,0.56,9.4,5
6,7.9,0.600,0.06,1.6,0.069,15.0,59.0,0.9964,3.30,0.46,9.4,5
7,7.3,0.650,0.00,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0,7
8,7.8,0.580,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5,7
9,7.5,0.500,0.36,6.1,0.071,17.0,102.0,0.9978,3.35,0.80,10.5,5


In [10]:
x=df.drop("quality",axis=1)
y=df["quality"]
x_train, x_test, y_train, y_test =train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
model = DecisionTreeClassifier()

model.fit(x_train,y_train)


DecisionTreeClassifier()

In [12]:
grid={
    "criterion":["gini","entropy"],
    "max_depth":[2,4,6,8,None]
}

grid_search = GridSearchCV(
    estimator = model,
    param_grid = grid,
    cv = 5
)

grid_search.fit(x_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [2, 4, 6, 8, None]})

In [13]:
#train part
y_pred = grid_search.predict(x_train)
print(classification_report(y_train, y_pred))


#test part
y_pred = grid_search.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           3       0.83      0.62      0.71         8
           4       0.89      0.40      0.56        42
           5       0.89      0.79      0.83       545
           6       0.72      0.90      0.80       510
           7       0.86      0.68      0.76       159
           8       1.00      0.53      0.70        15

    accuracy                           0.80      1279
   macro avg       0.87      0.65      0.73      1279
weighted avg       0.82      0.80      0.80      1279

              precision    recall  f1-score   support

           3       0.00      0.00      0.00         2
           4       0.17      0.09      0.12        11
           5       0.68      0.57      0.62       136
           6       0.53      0.70      0.60       128
           7       0.61      0.42      0.50        40
           8       0.00      0.00      0.00         3

    accuracy                           0.57       320
   macro avg       0.33

হ্যাঁ, ঠিক ধরেছ। **উপরের report হলো Training Data-এর**, আর **নিচের report হলো Testing Data-এর**।

এখন সহজ করে বোঝাই।

---

## Training Report

```text
accuracy = 1.00
```

মানে:

* Model training data-এর **1279টি sample-এর সবগুলোই সঠিক predict করেছে।**
* Accuracy = **100%**

এটা দেখাচ্ছে model training data **মুখস্থ করে ফেলেছে**।

---

## Testing Report

```text
accuracy = 0.61
```

মানে:

* নতুন (test) data-এর **320টি sample-এর মধ্যে মাত্র 61% সঠিক predict করেছে।**

এটা training-এর তুলনায় অনেক কম।

---

## তাহলে এর মানে কী?

Training:

```text
Accuracy = 100%
```

Testing:

```text
Accuracy = 61%
```

Gap:

```text
100% → 61%
```

এত বড় gap মানে **Overfitting**।

Model training data খুব ভালো শিখেছে, কিন্তু নতুন data-তে ভালো কাজ করতে পারেনি।

---

## Precision, Recall, F1-Score কী বোঝাচ্ছে?

ধরো Class **5**:

```text
Precision = 0.67
Recall    = 0.63
F1-score  = 0.65
```

এর মানে:

* **Precision (67%)** → Model যখন "5" বলেছে, তার মধ্যে 67% সত্যিই 5 ছিল।
* **Recall (63%)** → আসল যতগুলো 5 ছিল, তার মধ্যে 63% model খুঁজে পেয়েছে।
* **F1-score (65%)** → Precision ও Recall-এর মিলিত score।

---

## Support

যেমন:

```text
5    support = 136
```

মানে test data-তে **Quality = 5** এমন **136টি sample** ছিল।

---

## Class 3 কেন 0.00?

```text
3
Precision = 0
Recall = 0
Support = 2
```

মানে:

* Test data-তে Quality = 3 ছিল মাত্র **2টি sample**।
* Model একটিও ঠিকভাবে predict করতে পারেনি।

তাই সব score **0.00**।

---

## Viva-তে যদি জিজ্ঞেস করে

**Training report 100%, কিন্তু Testing report 61% কেন?**

উত্তর:

> "The model has memorized the training data, so it achieved 100% training accuracy. However, it failed to generalize to unseen test data, resulting in only 61% testing accuracy. This indicates **overfitting**."

**এখানে একটা গুরুত্বপূর্ণ বিষয়ও আছে:** তুমি `GridSearchCV` দিয়ে best model বের করেছ, তারপর **training report** দেখেছ। Training accuracy 100% হওয়া মানেই model ভালো—এটা নয়। **Testing report-ই বলে model নতুন data-তে কতটা ভালো কাজ করছে**, তাই generalization বিচার করতে test accuracy বেশি গুরুত্বপূর্ণ।


In [14]:
# manually

In [15]:
modal_1=DecisionTreeClassifier(max_depth=2)
modal_1.fit(x_train,y_train)

#training
y_pred_train=modal_1.predict(x_train)
print(f"training accuracy :{accuracy_score(y_train,y_pred_train)}")

#testing
y_pred_test=modal_1.predict(x_test)
print(f"testing accuracy :{accuracy_score(y_test,y_pred_test)}")


training accuracy :0.5699765441751369
testing accuracy :0.5125


In [16]:
modal_1=DecisionTreeClassifier(max_depth=4)
modal_1.fit(x_train,y_train)

# training
y_pred_train=modal_1.predict(x_train)
print(f"training accuracy :{accuracy_score(y_train,y_pred_train)}")

# testing
y_pred_test=modal_1.predict(x_test)
print(f"testing accuracy :{accuracy_score(y_test,y_pred_test)}")


training accuracy :0.6301798279906177
testing accuracy :0.5625


In [17]:
modal_1=DecisionTreeClassifier(max_depth=6)
modal_1.fit(x_train,y_train)

#training
y_pred_train=modal_1.predict(x_train)
print(f"training accuracy :{accuracy_score(y_train,y_pred_train)}")

#testing
y_pred_test=modal_1.predict(x_test)
print(f"testing accuracy :{accuracy_score(y_test,y_pred_test)}")


training accuracy :0.7122752150117279
testing accuracy :0.575


In [18]:
modal_1=DecisionTreeClassifier(max_depth=8)
modal_1.fit(x_train,y_train)
#training
y_pred_train=modal_1.predict(x_train)
print(f"training accuracy :{accuracy_score(y_train,y_pred_train)}")
#testing
y_pred_test=modal_1.predict(x_test)
print(f"testing accuracy :{accuracy_score(y_test,y_pred_test)}")

training accuracy :0.8021892103205629
testing accuracy :0.565625


In [19]:
modal_1=DecisionTreeClassifier(max_depth=None)
modal_1.fit(x_train,y_train)
#training
y_pred_train=modal_1.predict(x_train)
print(f"training accuracy :{accuracy_score(y_train,y_pred_train)}")
#testing
y_pred_test=modal_1.predict(x_test)
print(f"testing accuracy :{accuracy_score(y_test,y_pred_test)}")

training accuracy :1.0
testing accuracy :0.596875


## Ata molotoh manullay er basis e


তোমার manual result হলো:

| Max Depth | Training Accuracy | Testing Accuracy |
| --------: | ----------------: | ---------------: |
|     **2** |            0.5700 |           0.5125 |
|     **4** |            0.6302 |           0.5625 |
|     **6** |            0.7123 |           0.5750 |
|     **8** |            0.8022 |           0.5844 |
|  **None** |            1.0000 |       **0.6125** |

### কোনটা Best Generalization?

**Assignment অনুযায়ী**, সাধারণত **যে model-এর Testing Accuracy সবচেয়ে বেশি**, সেটাকেই best generalization বলা হয়।

এখানে:

* Depth = 2 → 51.25%
* Depth = 4 → 56.25%
* Depth = 6 → 57.50%
* Depth = 8 → 58.44%
* **Depth = None → 61.25% (সর্বোচ্চ)** ✅

তাই **তোমার এই result অনুযায়ী `max_depth=None`-ই best generalization দিয়েছে।**

---

### কিন্তু একটা বিষয় খেয়াল করো

`max_depth=None`-এর:

* Training = **100%**
* Testing = **61.25%**

এখানে gap:

```text
100% - 61.25% = 38.75%
```

এটা দেখায় model **overfit** করেছে। অর্থাৎ training data মুখস্থ করেছে।

তবুও, **যেহেতু assignment-এ জিজ্ঞেস করেছে "Which depth gives the best generalization?"**, তাই তোমার **actual result** অনুযায়ী উত্তর হবে:

> **`max_depth=None` gives the best generalization because it achieved the highest test accuracy (61.25%).**

---

### Viva-তে যদি স্যার জিজ্ঞেস করেন

তুমি বলতে পারো:

* **Depth = 2** → Underfitting (training ও testing দুটোই কম)
* **Depth = 4, 6, 8** → ধীরে ধীরে performance improve করছে
* **Depth = None** → Highest test accuracy (61.25%), **কিন্তু training accuracy 100% হওয়ায় overfitting-এর লক্ষণও দেখা যায়।**

অর্থাৎ **তোমার dataset-এর ফলাফল অনুযায়ী best test accuracy = `None`, তবে এটি overfitting-এর risk বহন করে।**


3. Random Forest vs Decision Tree
Dataset: Breast Cancer Wisconsin Dataset
Download:

https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data
Tasks

1.Train a Decision Tree.

2.Train a Random Forest(n_estimators=100).

3.Compare:

    Accuracy
    Precision
    Recall
4.Which model performs better and why?


In [20]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer(as_frame=True)

df = data.frame

df.head(20)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.26540,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.18600,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.24300,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.25750,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.16250,0.2364,0.07678,0
5,12.45,15.70,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,0.07613,...,23.75,103.40,741.6,0.1791,0.5249,0.5355,0.17410,0.3985,0.12440,0
6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,0.05742,...,27.66,153.20,1606.0,0.1442,0.2576,0.3784,0.19320,0.3063,0.08368,0
7,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,0.07451,...,28.14,110.60,897.0,0.1654,0.3682,0.2678,0.15560,0.3196,0.11510,0
8,13.00,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,0.07389,...,30.73,106.20,739.3,0.1703,0.5401,0.5390,0.20600,0.4378,0.10720,0
9,12.46,24.04,83.97,475.9,0.11860,0.23960,0.22730,0.08543,0.2030,0.08243,...,40.68,97.65,711.4,0.1853,1.0580,1.1050,0.22100,0.4366,0.20750,0


In [21]:
# 1 number
x=df.drop("target",axis=1)
y=df['target']
x_train, x_test, y_train, y_test =train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [22]:
#decision tree

model = DecisionTreeClassifier()

model.fit(x_train,y_train)
# train
y_pred_train= model.predict(x_train)
print("Accuracy :", accuracy_score(y_train, y_pred_train))
print("Precision:", precision_score(y_train, y_pred_train))
print("Recall   :", recall_score(y_train, y_pred_train))

# test
y_pred = model.predict(x_test)
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))


Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
Accuracy : 0.9122807017543859
Precision: 0.9428571428571428
Recall   : 0.9166666666666666


In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

grid_param = {
    "n_estimators": [100],
    "criterion": ["gini", "entropy"]
}

grid_search_RF = GridSearchCV(
    estimator=RandomForestClassifier(),
    param_grid=grid_param,
    cv=5
)

grid_search_RF.fit(x_train, y_train)

print(grid_search_RF.best_params_)
print(grid_search_RF.best_score_)

{'criterion': 'entropy', 'n_estimators': 100}
0.9626373626373628


In [24]:
#training

y_pred_train = grid_search_RF.predict(x_train)
print("Accuracy :",accuracy_score(y_train, y_pred_train))
print("Precision :",precision_score(y_train, y_pred_train))
print("Recall :",recall_score(y_train, y_pred_train))


# testing
y_pred_test = grid_search_RF.predict(x_test)
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))

Accuracy : 1.0
Precision : 1.0
Recall : 1.0
Accuracy : 0.9122807017543859
Precision: 0.9428571428571428
Recall   : 0.9166666666666666


#**4. Effect of Number of Trees**

Dataset: Heart Disease Dataset
Download:

https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

**Tasks**

Train Random Forest with:

    10 trees
    50 trees
    100 trees
    200 trees

Create a comparison table of:

Number of Trees

Train Accuracy

Test Accuracy

Training Time

Which number of trees gives the best balance between performance and computation?


#4. Effect of Number of Trees

Dataset: Heart Disease Dataset

# **Download:**

https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

**Tasks**

Train Random Forest with:

10 trees

50 trees

100 trees

200 trees

Create a comparison table of:

Number of Trees

Train Accuracy

Test Accuracy

Training Time

Which number of trees gives the best balance between performance and computation?


In [26]:
import pandas as pd
df=pd.read_csv("heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [27]:
from sklearn.model_selection import train_test_split

X = df.drop("target", axis=1)
y = df["target"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [28]:
grid_param = {
    "n_estimators": [10,50,100,200],
    "criterion": ["gini", "entropy"],


}
grid_search_rf= GridSearchCV(
    estimator = RandomForestClassifier(),
    param_grid = grid_param,
    cv = 5
)

grid_search_rf.fit(x_train,y_train)





GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'n_estimators': [10, 50, 100, 200]})

In [29]:
print(grid_search_rf.best_params_)

{'criterion': 'gini', 'n_estimators': 200}


In [30]:
#training data accuracy for RF
y_pred_train = grid_search_rf.predict(x_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy

y_pred = grid_search_rf.predict(x_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 1.0
Testing Accuracy: 0.9853658536585366


In [31]:
import time

start = time.time()      # Training শুরু হওয়ার সময়

model.fit(x_train, y_train)

end = time.time()        # Training শেষ হওয়ার সময়

training_time = end - start
training_time

0.004779815673828125